# KAN-Vim — Thử nghiệm sơ bộ (pilot) trên Colab T4
**Trước khi chạy:** upload `kan-vim.zip` mới lên Drive (đè bản cũ) · *Thời gian chạy -> Thay đổi loại thời gian chạy -> T4 GPU* · chạy lần lượt Cell 1 -> 5.

Mọi kết quả lưu vào `MyDrive/kan-vim-results/pilot`. Nếu Colab ngắt giữa chừng: chạy lại Cell 1 rồi Cell 4 — các lần chạy đã xong sẽ được **bỏ qua**, chỉ chạy tiếp phần còn thiếu.

## Cell 1 — Chuẩn bị (GPU, code, checkpoint, CIFAR-100 từ Drive)
Mong đợi: `cuda ok: True`, có `vim_pure.py`, `ckpt ok`, `CIFAR-100 ok`, `41 passed`.

In [ ]:
import os, sys, glob, shutil, subprocess, torch
print('torch', torch.__version__, '| cuda ok:', torch.cuda.is_available())
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
RES = f'{DRIVE}/kan-vim-results/pilot'
os.makedirs(RES, exist_ok=True)

# code
os.chdir('/content')
zips = sorted(glob.glob(f'{DRIVE}/**/kan-vim*.zip', recursive=True), key=os.path.getmtime)
assert zips, 'Không tìm thấy kan-vim.zip trên Drive'
print('zip:', zips[-1])
shutil.rmtree('/content/kan-vim', ignore_errors=True)
subprocess.run(['unzip', '-q', '-o', zips[-1], '-d', '/content'], check=True)
os.chdir('/content/kan-vim'); sys.path.insert(0, '/content/kan-vim')
print('kanvim:', sorted(os.listdir('kanvim')))

# checkpoint
from huggingface_hub import hf_hub_download
shutil.copy(hf_hub_download('hustvl/Vim-tiny-midclstok', 'vim_t_midclstok_76p1acc.pth'), '/content/kan-vim/vim_t.pth')
print('ckpt ok')

# CIFAR-100: lấy từ Drive nếu đã có, nếu chưa thì tải 1 lần rồi lưu lên Drive
DDATA = f'{DRIVE}/kan-vim-data'
if os.path.isdir(f'{DDATA}/cifar-100-python'):
    shutil.copytree(f'{DDATA}/cifar-100-python', '/content/data/cifar-100-python', dirs_exist_ok=True)
else:
    from torchvision import datasets
    datasets.CIFAR100('/content/data', download=True)
    shutil.copytree('/content/data/cifar-100-python', f'{DDATA}/cifar-100-python', dirs_exist_ok=True)
print('CIFAR-100 ok')

out = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True)
print((out.stdout + out.stderr).strip().splitlines()[-1])

## Cell 2 — Kiểm tra Vim thuần PyTorch (ảnh chó Samoyed) — mong đợi **ĐÚNG**

In [ ]:
import os, sys, urllib.request, torch
os.chdir('/content/kan-vim'); sys.path.insert(0, '/content/kan-vim')
from PIL import Image
from torchvision import transforms as T
from kanvim.vim_pure import load_vim_pure
m = load_vim_pure('tiny', 'vim_t.pth', num_classes=1000).cuda().eval()
urllib.request.urlretrieve('https://github.com/pytorch/hub/raw/master/images/dog.jpg', '/content/dog.jpg')
tf = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
x = tf(Image.open('/content/dog.jpg').convert('RGB')).unsqueeze(0).cuda()
with torch.no_grad():
    top = m(x).softmax(-1)[0].topk(5)
print('top5:', top.indices.tolist(), [round(v, 3) for v in top.values.tolist()])
print('ĐÚNG' if 258 in top.indices.tolist() else 'SAI -> gửi output này cho Claude')
del m; torch.cuda.empty_cache()

## Cell 3 — Bảng hiệu năng: GMACs, tốc độ, VRAM cho mọi phương pháp × 75/50/25% (không cần train, ~10-20 phút)

In [ ]:
%cd /content/kan-vim
!python scripts/bench.py --backbone tiny --ckpt vim_t.pth --batch 64 --iters 3 --out {RES} 2>&1 | grep -v Warning

## Cell 4 — Fine-tune ngắn (pilot): CIFAR-100, 5.000 ảnh train, 2.000 ảnh test, 1 epoch, mức giữ 50%
Chạy theo thứ tự ưu tiên; mỗi lần chạy xong lưu ngay lên Drive. Thời gian mỗi lần chạy được in ra.
Hết giờ thì dừng lại cũng được — Cell 5 sẽ báo cáo những gì đã xong.

In [ ]:
import os, subprocess, time
os.chdir('/content/kan-vim')
PLAN = [('identity', 1.0), ('kan_prune', 0.5), ('mlp_prune', 0.5), ('kan_pool', 0.5),
        ('mlp_pool', 0.5), ('avg_pool', 0.5), ('max_pool', 0.5)]
COMMON = (f'--dataset cifar100 --data_root /content/data --backbone tiny --ckpt vim_t.pth --epochs 1 '
          f'--train_subset 5000 --test_subset 2000 --batch 32 --grad_ckpt --no_bench --seed 0 --out {RES}')
for r, q in PLAN:
    if os.path.exists(f'{RES}/cifar100_tiny_{r}_r{q}_s0.json'):
        print('đã có, bỏ qua:', r, q)
        continue
    t = time.time()
    print(f'>>> {r} keep={q}', flush=True)
    p = subprocess.run(f'python train.py {COMMON} --reducer {r} --ratio {q}', shell=True,
                       capture_output=True, text=True)
    lines = [l for l in (p.stdout + p.stderr).splitlines() if 'arning' not in l and 'warn' not in l]
    for l in lines[-15:]:
        print('   ', l)
    print(f'    xong sau {(time.time() - t) / 60:.1f} phút', flush=True)

## Cell 5 — Báo cáo: bảng + biểu đồ (lưu `report.md`, `fig_*.png` trong `MyDrive/kan-vim-results/pilot`)

In [ ]:
%cd /content/kan-vim
!python scripts/report.py {RES} > /dev/null
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{RES}/report.md', encoding='utf-8').read().replace('![efficiency](fig_efficiency.png)', '').replace('![tradeoff](fig_tradeoff.png)', '')))
for f in ['fig_efficiency.png', 'fig_tradeoff.png']:
    if os.path.exists(f'{RES}/{f}'): display(Image(f'{RES}/{f}'))